<a href="https://github.com/Mahendra2409/PyBlender/blob/main/Colab_Script/ccylinder_scaled_PC_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Mount Drive
from google.colab import drive
drive.mount('/content/drive')


## 1.Install Dependencies

In [ ]:
# @title
# 1. Download and extract Blender 4.0.2
!wget -nc https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz

# 2. Install dependencies into Blender 4.0's bundled Python
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m ensurepip
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install scipy matplotlib numpy blendertoolbox

In [ ]:
# @title
# 3. Download BlenderToolbox
!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
# Move the actual toolbox folder into your main /content/ directory so your script can 'see' it

!mv BlenderToolbox/blendertoolbox /content/
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install blendertoolbox

In [ ]:
#@title 2.1 Set Rendering Configuration Here
%%writefile render.py

# ==========================================
# 1. MAIN CONFIGURATION
# ==========================================

CONFIG = {
    # --- Output Controls ---
    "SAVE_BLEND_FILE": False,   # Set to True to save the .blend scene file alongside the image
    "FORCE_OVERWRITE": False,   # Set to True to re-render even if the output image already exists

    # --- Paths ---
    "DRIVE_BASE_PATH": "/content/drive/MyDrive/PyBlender_Render_Farm",
    "PC_TYPE": "ccylinder_scaled_PC_new",
    "GT_FILENAME": "gt_ccylinder.xyz",
    "COLORMAP": "turbo",

    "LOCAL_COLAB_BASE": "/content/local_data",

    # --- Blender Render Settings ---
    "IMG_RES_X": 1000,
    "IMG_RES_Y": 1000,
    "NUM_SAMPLES": 100,
    "EXPOSURE": 1.5,

    # --- Point Cloud Settings ---
    "PT_SIZE": 0.009,
    "PT_COLOR": [0.5, 1.0, 1.0, 0.0, 0.0],

    # --- Object Transforms ---
    "OBJ_LOCATION": (-0.370186, -5.31554, -2.55748),
    "OBJ_ROTATION": (448.706, -0.984887, -94.562),
    "OBJ_SCALE": (3.71272, 3.71272, 3.71272),

    # --- Camera Settings ---
    "CAM_LOCATION": (0.141656, 2.50472, 1.37419),
    "LOOK_AT": (0, 0, 0),
    "FOCAL_LENGTH": 45,

    # --- Light Settings ---
    "LIGHT_ANGLE": (6, -30, -155),
    "LIGHT_STRENGTH": 2,
    "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05
}

import os
import shutil
import bpy

# --- THE FIX FOR COLAB/MATPLOTLIB ---
os.environ['MPLBACKEND'] = 'Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# ------------------------------------

import numpy as np
from scipy.spatial import cKDTree
import blendertoolbox as bt



# ==========================================
# 2. GROUND TRUTH OVERRIDES
# ==========================================
GT_OVERRIDES = {
    # Example: Uncomment to apply only to GT
    # "PT_SIZE": 0.012,
}

# ==========================================
# 3. DERIVED PATHS
# ==========================================
DRIVE_SOURCE_DIR = os.path.join(CONFIG["DRIVE_BASE_PATH"], "PointCloud", "xyzFormat", CONFIG["PC_TYPE"])
DRIVE_OUTPUT_DIR = os.path.join(CONFIG["DRIVE_BASE_PATH"], "RenderImages", CONFIG["PC_TYPE"], f"{CONFIG['COLORMAP']}_colormap")

LOCAL_SOURCE_DIR = os.path.join(CONFIG["LOCAL_COLAB_BASE"], CONFIG["PC_TYPE"])
LOCAL_GROUND_TRUTH_PATH = os.path.join(LOCAL_SOURCE_DIR, CONFIG["GT_FILENAME"])


# ==========================================
# 4. PIPELINE FUNCTIONS
# ==========================================
def setup_colab_environment():
    print("--- Setting up Environment ---")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

    if os.path.exists(LOCAL_SOURCE_DIR):
        shutil.rmtree(LOCAL_SOURCE_DIR)

    print(f"Copying point clouds from Drive to Local Colab Storage...")
    shutil.copytree(DRIVE_SOURCE_DIR, LOCAL_SOURCE_DIR)
    print("Copy complete!\n")


def render_point_clouds():
    print("--- Starting Render Process ---")

    ground_truth_points = np.loadtxt(LOCAL_GROUND_TRUTH_PATH)
    tree = cKDTree(ground_truth_points)

    for filename in os.listdir(LOCAL_SOURCE_DIR):
        if not filename.endswith(".xyz"):
            continue

        # 1. Determine if this specific file is the Ground Truth
        is_gt = (filename == CONFIG["GT_FILENAME"])

        # 2. Create the Active Config
        active_cfg = CONFIG.copy()
        if is_gt:
            active_cfg.update(GT_OVERRIDES)

        # 3. Output checking (UPDATED WITH FORCE_OVERWRITE FLAG)
        render_output = os.path.join(DRIVE_OUTPUT_DIR, f"{os.path.splitext(filename)[0]}.png")
        if os.path.exists(render_output) and not active_cfg["FORCE_OVERWRITE"]:
            print(f"Output already exists: {filename}. Skipping...")
            continue

        print(f"Processing: {filename} {'(GROUND TRUTH MODE)' if is_gt else ''}")

        # 4. Initialize Blender
        bt.blenderInit(
            active_cfg["IMG_RES_X"],
            active_cfg["IMG_RES_Y"],
            active_cfg["NUM_SAMPLES"],
            active_cfg["EXPOSURE"]
        )

        # 5. Load Points
        current_points = ground_truth_points if is_gt else np.loadtxt(os.path.join(LOCAL_SOURCE_DIR, filename))

        # 6. Apply Color Logic
        if is_gt:
            colormap = plt.get_cmap(active_cfg["COLORMAP"])
            min_color = colormap(0.0)[:3]
            colors = np.tile(min_color, (current_points.shape[0], 1))
        else:
            distances, _ = tree.query(current_points)
            max_distance = np.percentile(distances, 99)
            normalized_distances = np.clip(distances / max_distance, 0, 1)
            normalized_distances = np.log1p(normalized_distances) / np.log1p(1)
            colormap = plt.get_cmap(active_cfg["COLORMAP"])
            colors = colormap(normalized_distances)[:, :3]

        # 7. Setup Mesh and Materials
        mesh = bt.readNumpyPoints(
            current_points,
            active_cfg["OBJ_LOCATION"],
            active_cfg["OBJ_ROTATION"],
            active_cfg["OBJ_SCALE"]
        )
        mesh = bt.setPointColors(mesh, colors)

        ptColor = bt.colorObj(active_cfg["PT_COLOR"], 0.5, 1.0, 1.0, 0.0, 0.0)
        bt.setMat_pointCloudColored(mesh, ptColor, active_cfg["PT_SIZE"])

        # 8. Setup Environment
        cam = bt.setCamera(active_cfg["CAM_LOCATION"], active_cfg["LOOK_AT"], active_cfg["FOCAL_LENGTH"])
        sun = bt.setLight_sun(active_cfg["LIGHT_ANGLE"], active_cfg["LIGHT_STRENGTH"], active_cfg["SHADOW_SOFTNESS"])
        bt.setLight_ambient(color=active_cfg["AMBIENT_COLOR"])
        bt.shadowThreshold(alphaThreshold=active_cfg["SHADOW_THRESHOLD"], interpolationMode='CARDINAL')

        bpy.context.scene.use_nodes = True
        tree_nodes = bpy.context.scene.node_tree
        tree_nodes.nodes.clear()

        render_layers = tree_nodes.nodes.new('CompositorNodeRLayers')
        denoise_node = tree_nodes.nodes.new(type='CompositorNodeDenoise')
        composite = tree_nodes.nodes.new('CompositorNodeComposite')
        viewer = tree_nodes.nodes.new('CompositorNodeViewer')

        render_layers.location = (-300, 0)
        denoise_node.location = (0, 0)
        composite.location = (300, 0)
        viewer.location = (300, -200)

        tree_nodes.links.new(render_layers.outputs['Image'], denoise_node.inputs['Image'])
        tree_nodes.links.new(denoise_node.outputs['Image'], composite.inputs['Image'])
        tree_nodes.links.new(denoise_node.outputs['Image'], viewer.inputs['Image'])

        # 9. Save Colormap Gradient
        colormap_path = os.path.join(DRIVE_OUTPUT_DIR, f"{active_cfg['COLORMAP']}_colormap.png")
        if not os.path.exists(colormap_path):
            gradient = np.linspace(0, 1, 256).reshape(1, -1)
            gradient = np.vstack([gradient] * 50)
            plt.figure(figsize=(6, 1))
            plt.imshow(gradient, aspect="auto", cmap=active_cfg["COLORMAP"])
            plt.axis("off")
            plt.savefig(colormap_path, bbox_inches="tight", pad_inches=0, dpi=300)
            plt.close()
            print(f"Colormap saved at {colormap_path}")

        # 10. Save Blend File (UPDATED LOGIC)
        if active_cfg["SAVE_BLEND_FILE"]:
            blend_file_name = f"{os.path.splitext(filename)[0]}.blend"
            blend_file_path = os.path.join(DRIVE_OUTPUT_DIR, blend_file_name)
            bpy.ops.wm.save_mainfile(filepath=blend_file_path)
            print(f"Blend file saved at {blend_file_path}")

        # 11. Render
        bt.renderImage(render_output, cam)
        print(f"Render complete for {filename}\n")


if __name__ == "__main__":
    setup_colab_environment()
    render_point_clouds()

In [ ]:
#@title 2.3 Start Rendering Render

!./blender-4.0.2-linux-x64/blender -b -P render.py

In [ ]:
#@title Extra: 3D PointCloud Visualizer
import numpy as np
import plotly.graph_objs as go

# 1. PATH TO YOUR POINT CLOUD
# Point this to any .xyz file you want to inspect (either on Drive or Local Colab storage)
PC_PATH = "/content/local_data/ccylinder_scaled_PC_new/AD_ccylinder_gaussian_3.5.xyz"

def view_point_cloud(filepath, max_points=100000):
    print(f"Loading {filepath}...")
    try:
        points = np.loadtxt(filepath)
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # 2. OPTIMIZATION
    # Browsers will crash if you try to render millions of points in Plotly.
    # If the point cloud is huge, we randomly sample it down to keep it interactive.
    if len(points) > max_points:
        print(f"Subsampling from {len(points)} to {max_points} points for smooth viewing...")
        indices = np.random.choice(len(points), max_points, replace=False)
        points = points[indices]

    x, y, z = points[:, 0], points[:, 1], points[:, 2]

    # Calculate center bounding box for reference
    center_x = (np.max(x) + np.min(x)) / 2
    center_y = (np.max(y) + np.min(y)) / 2
    center_z = (np.max(z) + np.min(z)) / 2
    print(f"\n--- Point Cloud Stats ---")
    print(f"X range: {np.min(x):.2f} to {np.max(x):.2f} (Center: {center_x:.2f})")
    print(f"Y range: {np.min(y):.2f} to {np.max(y):.2f} (Center: {center_y:.2f})")
    print(f"Z range: {np.min(z):.2f} to {np.max(z):.2f} (Center: {center_z:.2f})")

    # 3. BUILD THE INTERACTIVE PLOT
    print("\nGenerating interactive 3D viewer...")
    fig = go.Figure(data=[go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=1.5,
            color=z,                 # Color points by height to give depth
            colorscale='Viridis',
            opacity=0.8
        )
    )])

    # Lock the aspect ratio so the object doesn't stretch weirdly
    fig.update_layout(
        scene=dict(
            aspectmode='data',
            xaxis_title='X Axis',
            yaxis_title='Y Axis',
            zaxis_title='Z Axis'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        height=700
    )

    fig.show()

# Run the viewer
view_point_cloud(PC_PATH)